# TEP Process Fault Classification and Anomaly Detection Using Signal-Quality-Aware Features

> **PCA copy** of `TEP_Fault_Detection_optimized.ipynb`. This notebook is identical to the optimized version with one methodological addition: **PCA dimensionality reduction** applied to the Wavelet and Combined feature sets. PCA is fit on training data only (fold-safe in CV), applied after scaling, and retains 95% explained variance. The Statistical feature baseline is preserved unchanged.

**Author:** Rishabh Goenka
**Course:** EE 344 — Data-Driven Modeling and Machine Learning (Winter 2026)
**Dataset:** Tennessee Eastman Process (TEP) Simulation — Rieth et al. (2017)

---

## Problem Statement

Modern industrial plants rely on dozens of sensors to monitor temperature, pressure, flow rates, and chemical concentrations. The Tennessee Eastman Process (TEP) benchmark simulates 20 distinct process fault types across 52 variables recorded at 3-minute intervals, providing a challenging testbed for fault detection and classification.

This notebook implements two complementary tasks:
1. **Binary anomaly detection** — distinguishing normal operation from any fault condition.
2. **Multi-class fault classification** — identifying which of 20 fault types is present.

Three feature extraction pipelines are compared:
- **Statistical features** — mean, std, skewness, kurtosis, min, max, plus signal-quality indicators (coefficient of variation, lag-1 autocorrelation, first-difference std, high-frequency energy ratio).
- **Wavelet features** — multi-level DWT decomposition with energy, entropy, and magnitude.
- **Combined** — union of statistical and wavelet features.

These are evaluated across **supervised classifiers** (Logistic Regression, Random Forest, XGBoost) and **one-class methods** (Isolation Forest, Autoencoder) trained only on normal data.

**Primary metric:** Macro F1-score
**Evaluation:** Run-level GroupKFold 5-fold cross-validation with proper group separation.

### Methodology Note

The original project proposal framed this as "sensor degradation detection via signal quality analysis." After careful analysis, the TEP fault types are **process-level disturbances** (step changes in feed composition, random variation in process parameters, valve sticking, etc.), not sensor-specific degradation events (increased noise, baseline drift, stuck readings). While signal-quality features (variance structure, autocorrelation, spectral content) are included as complementary descriptors, the primary detection mechanism relies on changes in process variable behavior. The results should be interpreted as **process fault classification performance**, not sensor degradation detection. See `AUDIT_AND_REPAIR.md` for a detailed discussion.

## 1) Setup

In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time

import pywt
from scipy.stats import skew, kurtosis

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import xgboost as xgb

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, roc_curve, roc_auc_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
DATA_DIR = "data"

WINDOW_SIZE = 30        # 30 samples = 90 min at 3-min intervals
STRIDE_TRAIN = WINDOW_SIZE  # non-overlapping windows for training (reduces redundancy ~3x)
STRIDE_TEST = 10            # dense stride for evaluation (preserves visualization resolution)

WAVELET = "db4"         # Daubechies-4
WAVELET_LEVEL = 2       # 2 decomposition levels (appropriate for window of 30)

N_FOLDS = 5

FAULT_START_TRAIN = 20  # 1-indexed sample where fault begins in training runs
FAULT_START_TEST = 160  # 1-indexed sample where fault begins in testing runs

SUBSAMPLE_RUNS = 30     # runs per fault type (None = all 500)

MEASURED_ONLY = True     # restrict to xmeas variables (exclude xmv controller outputs)

PCA_VARIANCE = 0.95     # retained explained variance for PCA (Wavelet and Combined)

## 2) Load Data

The TEP dataset consists of four parquet files:
- **TEP_FaultFree_Training** — 500 normal runs, 500 samples each
- **TEP_FaultFree_Testing** — 500 normal runs, 960 samples each
- **TEP_Faulty_Training** — 20 fault types × 500 runs × 500 samples
- **TEP_Faulty_Testing** — 20 fault types × 500 runs × 960 samples

Each sample contains 52 process variables: 41 measured (`xmeas`) and 11 manipulated (`xmv`), recorded at 3-minute intervals. **We restrict to the 41 measured variables only**, excluding manipulated variables (controller outputs) that would confound fault detection by revealing control-loop responses rather than sensor behavior.

In [ ]:
print("Loading parquet files ...")
t0 = time.time()
ff_train = pd.read_parquet(os.path.join(DATA_DIR, "TEP_FaultFree_Training.parquet"))
ff_test  = pd.read_parquet(os.path.join(DATA_DIR, "TEP_FaultFree_Testing.parquet"))
f_train  = pd.read_parquet(os.path.join(DATA_DIR, "TEP_Faulty_Training.parquet"))
f_test   = pd.read_parquet(os.path.join(DATA_DIR, "TEP_Faulty_Testing.parquet"))
print(f"Loaded in {time.time() - t0:.1f}s")

for name, df in [("FaultFree Train", ff_train), ("FaultFree Test", ff_test),
                 ("Faulty Train", f_train), ("Faulty Test", f_test)]:
    print(f"  {name:16s}: shape={df.shape}")

In [ ]:
meta_cols = ["faultNumber", "simulationRun", "sample"]
all_sensor_cols = [c for c in f_train.columns if c not in meta_cols]

if MEASURED_ONLY:
    sensor_cols = [c for c in all_sensor_cols if c.startswith("xmeas")]
else:
    sensor_cols = all_sensor_cols

print(f"Using {len(sensor_cols)} sensor columns (MEASURED_ONLY={MEASURED_ONLY})")
print(f"  First 5: {sensor_cols[:5]}")
print(f"  Last 3:  {sensor_cols[-3:]}")

if "faultNumber" not in ff_train.columns:
    ff_train["faultNumber"] = 0
    ff_test["faultNumber"] = 0
else:
    ff_train["faultNumber"] = ff_train["faultNumber"].fillna(0).astype(int)
    ff_test["faultNumber"] = ff_test["faultNumber"].fillna(0).astype(int)

train_df = pd.concat([ff_train, f_train], ignore_index=True)
test_df  = pd.concat([ff_test,  f_test],  ignore_index=True)

del ff_train, ff_test, f_train, f_test
gc.collect()

for col in meta_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype(int)
        test_df[col]  = test_df[col].astype(int)

if SUBSAMPLE_RUNS is not None:
    rng = np.random.RandomState(RANDOM_STATE)
    def subsample_runs(df, n, rng):
        parts = []
        for fn in sorted(df["faultNumber"].unique()):
            sub = df[df["faultNumber"] == fn]
            runs = sub["simulationRun"].unique()
            if len(runs) > n:
                runs = rng.choice(runs, n, replace=False)
                sub = sub[sub["simulationRun"].isin(runs)]
            parts.append(sub)
        return pd.concat(parts, ignore_index=True)

    train_df = subsample_runs(train_df, SUBSAMPLE_RUNS, rng)
    test_df  = subsample_runs(test_df,  SUBSAMPLE_RUNS, rng)
    print(f"Subsampled to {SUBSAMPLE_RUNS} runs per fault type")

print(f"Training shape: {train_df.shape}")
print(f"Testing shape:  {test_df.shape}")

## 3) Data Audit

Verify data integrity: missing values, label distribution, simulation run counts.

In [ ]:
print("=== Missing Values ===")
print(f"Training NaN count:  {train_df.isna().sum().sum()}")
print(f"Testing NaN count:   {test_df.isna().sum().sum()}")
print(f"Training Inf count:  {np.isinf(train_df[sensor_cols].values).sum()}")
print(f"Testing Inf count:   {np.isinf(test_df[sensor_cols].values).sum()}")
print(f"Training duplicates: {train_df.duplicated().sum()}")

print()
print("=== Fault Types & Run Counts (Training) ===")
ft = train_df.groupby("faultNumber").agg(
    Runs=("simulationRun", "nunique"), Samples=("sample", "count"))
print(ft.to_string())

print()
print(f"Sensor columns: {len(sensor_cols)} ({'measured only' if MEASURED_ONLY else 'all 52'})")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
counts = train_df["faultNumber"].value_counts().sort_index()
colors = ["#2ecc71" if fn == 0 else "#e74c3c" if fn in [3, 9, 15] else "#3498db"
          for fn in counts.index]
ax.bar(counts.index.astype(str), counts.values, color=colors)
ax.set_xlabel("Fault Type")
ax.set_ylabel("Sample Count")
ax.set_title("Training Data — Samples per Fault Type (green=normal, red=hard faults)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4) Exploratory Data Analysis

Visualize sensor behavior under normal and faulty conditions.

In [ ]:
run0 = train_df[train_df["faultNumber"] == 0]["simulationRun"].unique()[0]
normal_run = train_df[(train_df["faultNumber"] == 0) & (train_df["simulationRun"] == run0)]

fault2_run_id = train_df[train_df["faultNumber"] == 2]["simulationRun"].unique()[0]
fault2_run = train_df[(train_df["faultNumber"] == 2) & (train_df["simulationRun"] == fault2_run_id)]

plot_sensors = sensor_cols[:4]
fig, axes = plt.subplots(len(plot_sensors), 1, figsize=(12, 3 * len(plot_sensors)), sharex=True)
for idx, sensor in enumerate(plot_sensors):
    axes[idx].plot(normal_run["sample"].values, normal_run[sensor].values, label="Normal", alpha=0.7)
    axes[idx].plot(fault2_run["sample"].values, fault2_run[sensor].values, label="Fault 2", alpha=0.7)
    axes[idx].axvline(x=FAULT_START_TRAIN, color="red", linestyle="--", alpha=0.5, label="Fault onset")
    axes[idx].set_ylabel(sensor)
    axes[idx].legend(loc="upper right", fontsize=8)
    axes[idx].grid(True, alpha=0.3)
axes[-1].set_xlabel("Sample")
fig.suptitle("Time Series: Normal vs Fault 2", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr = train_df[train_df["faultNumber"] == 0][sensor_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Sensor Correlation Matrix (Normal Operation)")
ax.set_xticks(range(len(sensor_cols)))
ax.set_yticks(range(len(sensor_cols)))
ax.set_xticklabels(sensor_cols, rotation=90, fontsize=5)
ax.set_yticklabels(sensor_cols, fontsize=5)
plt.tight_layout()
plt.show()

In [ ]:
n_pca_sample = min(20000, len(train_df))
pca_idx = np.random.choice(len(train_df), n_pca_sample, replace=False)
X_pca = train_df.iloc[pca_idx][sensor_cols].values
y_pca = train_df.iloc[pca_idx]["faultNumber"].values

pca = PCA(n_components=2)
X_2d = pca.fit_transform(StandardScaler().fit_transform(X_pca))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mask_normal = y_pca == 0
mask_fault  = y_pca != 0
axes[0].scatter(X_2d[mask_normal, 0], X_2d[mask_normal, 1], s=2, alpha=0.3, label="Normal", c="green")
axes[0].scatter(X_2d[mask_fault, 0],  X_2d[mask_fault, 1],  s=2, alpha=0.1, label="Faulty", c="red")
axes[0].set_title("PCA — Normal vs Faulty")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[0].legend()

for fn in sorted(np.unique(y_pca)):
    mask = y_pca == fn
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], s=2, alpha=0.15, label=str(fn))
axes[1].set_title("PCA — By Fault Type")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[1].legend(fontsize=6, ncol=3, markerscale=3)

plt.tight_layout()
plt.show()

### Key EDA Observations

- The measured sensors show varying degrees of correlation — strong blocks of correlated measurements exist, motivating dimensionality reduction.
- Time series plots show that some faults produce obvious shifts or increased variance, while others (notably faults 3, 9, 15) are much more subtle.
- The PCA scatter plot reveals partial cluster separation for some fault types, but significant overlap remains for the hard faults.
- Class balance: each fault type has equal representation, but the binary split (normal vs. fault) is heavily imbalanced toward faulty data.

## 5) Windowing and Feature Extraction

### Approach

Each simulation run is segmented into overlapping time windows. From each window, three feature sets are extracted:

| Pipeline | Description |
|----------|-------------|
| Statistical | Level features (mean, std, skew, kurtosis, min, max) + signal-quality features (coefficient of variation, lag-1 autocorrelation, first-difference std, high-frequency energy ratio) per sensor |
| Wavelet | Energy, entropy, mean magnitude from 2-level db4 DWT per sensor |
| Combined | Union of statistical and wavelet features |

**Signal-quality features rationale:** While the primary detection mechanism for TEP faults relies on process-level changes, signal-quality features capture complementary information about the variability structure, temporal correlation, and frequency content of sensor readings within each window.

**Window labeling:** A window from a faulty run is labeled as faulty only if more than half its samples are post-fault-injection.

### Stride Optimization (this copy only)
- **Training** uses `STRIDE_TRAIN = WINDOW_SIZE` (non-overlapping) to eliminate redundant, heavily-overlapping windows. This reduces training set size ~3× without losing coverage.
- **Testing** uses `STRIDE_TEST = 10` (dense overlap, matching original) to preserve evaluation resolution and visualization quality.
- This change does **not** introduce leakage: each window still comes from a single run, and the feature definitions are unchanged.

### Leakage Prevention
- Features are extracted per-run; no windows span across runs.
- Cross-validation groups use unique run identifiers `(faultNumber × 1000 + simulationRun)` so all windows from one simulation run always stay in the same fold.

In [ ]:
def extract_features(df, sensor_cols, window_size, stride, fault_start):
    """Sliding-window feature extraction with statistical and wavelet pipelines.

    Returns: (stat_features, wav_features, labels, group_ids, run_fault_types)
    """
    n_sensors = len(sensor_cols)
    stat_feats, wav_feats = [], []
    labels, group_ids, run_faults = [], [], []

    groups = list(df.groupby(["faultNumber", "simulationRun"]))
    n_groups = len(groups)
    t0 = time.time()

    for i, ((fault_num, run_id), group) in enumerate(groups):
        if (i + 1) % 200 == 0 or (i + 1) == n_groups:
            print(f"  Run {i+1}/{n_groups}  ({time.time()-t0:.0f}s)")

        group = group.sort_values("sample")
        values = group[sensor_cols].values.astype(np.float64)
        samples = group["sample"].values
        n = len(values)

        unique_gid = int(fault_num) * 1000 + int(run_id)

        for start in range(0, n - window_size + 1, stride):
            w = values[start:start + window_size]
            s = samples[start:start + window_size]

            if fault_num == 0:
                label = 0
            else:
                label = int(fault_num) if np.sum(s >= fault_start) > window_size // 2 else 0

            # --- Statistical features ---
            w_mean = np.mean(w, axis=0)
            w_std  = np.std(w, axis=0) + 1e-10
            w_skew = np.nan_to_num(skew(w, axis=0), nan=0.0)
            w_kurt = np.nan_to_num(kurtosis(w, axis=0), nan=0.0)
            w_min  = np.min(w, axis=0)
            w_max  = np.max(w, axis=0)

            # Signal-quality features
            coeff_var = w_std / (np.abs(w_mean) + 1e-10)

            # Lag-1 autocorrelation per sensor
            w_centered = w - w_mean
            numer = np.sum(w_centered[:-1] * w_centered[1:], axis=0)
            denom = np.sum(w_centered ** 2, axis=0) + 1e-10
            autocorr_lag1 = numer / denom

            # First-difference standard deviation
            diff_std = np.std(np.diff(w, axis=0), axis=0)

            # High-frequency energy ratio (proxy via first-diff energy / total energy)
            diff_energy = np.sum(np.diff(w, axis=0) ** 2, axis=0)
            total_energy = np.sum(w ** 2, axis=0) + 1e-10
            hf_ratio = diff_energy / total_energy

            sf = np.concatenate([
                w_mean, w_std, w_skew, w_kurt, w_min, w_max,
                coeff_var, autocorr_lag1, diff_std, hf_ratio,
            ])

            # --- Wavelet features ---
            coeffs = pywt.wavedec(w, WAVELET, level=WAVELET_LEVEL, axis=0)
            wf_parts = []
            for c in coeffs:
                energy   = np.sum(c ** 2, axis=0)
                c_abs    = np.abs(c)
                tot      = np.sum(c_abs, axis=0) + 1e-10
                c_norm   = c_abs / tot
                entropy  = -np.sum(c_norm * np.log(c_norm + 1e-10), axis=0)
                mean_mag = np.mean(c_abs, axis=0)
                wf_parts.extend([energy, entropy, mean_mag])

            stat_feats.append(sf)
            wav_feats.append(np.nan_to_num(np.concatenate(wf_parts), nan=0.0, posinf=0.0, neginf=0.0))
            labels.append(label)
            group_ids.append(unique_gid)
            run_faults.append(int(fault_num))

    return (np.array(stat_feats, dtype=np.float32),
            np.array(wav_feats,  dtype=np.float32),
            np.array(labels,     dtype=np.int32),
            np.array(group_ids,  dtype=np.int32),
            np.array(run_faults, dtype=np.int32))

In [ ]:
print(f"Extracting features from TRAINING data (stride={STRIDE_TRAIN}) ...")
X_stats_train, X_wav_train, y_train, group_ids_train, run_faults_train = \
    extract_features(train_df, sensor_cols, WINDOW_SIZE, STRIDE_TRAIN, FAULT_START_TRAIN)

print()
print(f"Extracting features from TESTING data (stride={STRIDE_TEST}) ...")
X_stats_test, X_wav_test, y_test, group_ids_test, run_faults_test = \
    extract_features(test_df, sensor_cols, WINDOW_SIZE, STRIDE_TEST, FAULT_START_TEST)

X_comb_train = np.hstack([X_stats_train, X_wav_train])
X_comb_test  = np.hstack([X_stats_test,  X_wav_test])

del train_df, test_df
gc.collect()

print()
print(f"Training windows: {len(y_train)}")
print(f"Testing windows:  {len(y_test)}")
print(f"Unique training groups: {len(np.unique(group_ids_train))}")

In [ ]:
for name, X in [("Stats train", X_stats_train), ("Wav train", X_wav_train),
                ("Comb train", X_comb_train), ("Stats test", X_stats_test),
                ("Wav test", X_wav_test), ("Comb test", X_comb_test)]:
    n_nan = np.isnan(X).sum()
    n_inf = np.isinf(X).sum()
    if n_nan > 0 or n_inf > 0:
        print(f"  {name:12s} had {n_nan} NaN, {n_inf} Inf — replacing with 0")
        np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"  {name:12s} shape={str(X.shape):20s}  [OK]")

print()
print("Label distribution (training windows):")
unique, counts = np.unique(y_train, return_counts=True)
dist_df = pd.DataFrame({"Fault": unique, "Windows": counts})
dist_df["Pct"] = (100 * dist_df["Windows"] / dist_df["Windows"].sum()).round(2)
print(dist_df.to_string(index=False))

## 6) Supervised Modeling — Binary Classification (Normal vs Fault)

Binary labels: 0 = normal, 1 = any fault. Evaluated with run-level GroupKFold (5-fold) where `StandardScaler` is fit on each fold's training split only (no data leakage).

**Group key:** Each group is a unique `(faultNumber, simulationRun)` pair, ensuring all windows from one simulation run stay in the same fold. This prevents temporal leakage from overlapping sliding windows.

**PCA addition:** For Wavelet+PCA and Combined+PCA feature sets, PCA (retaining 95% explained variance) is applied after scaling within each CV fold — fit on the fold's training partition only. Statistical features are left unchanged as a baseline.

In [ ]:
sup_models_binary = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                  random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             xgb.XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                             eval_metric="logloss", tree_method="hist",
                                             verbosity=0),
}

sup_models_multi = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                  random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             xgb.XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                             eval_metric="mlogloss", tree_method="hist",
                                             verbosity=0),
}

feat_train = {
    "Statistical": X_stats_train, "Wavelet": X_wav_train, "Combined": X_comb_train,
    "Wavelet+PCA": X_wav_train, "Combined+PCA": X_comb_train,
}
feat_test = {
    "Statistical": X_stats_test, "Wavelet": X_wav_test, "Combined": X_comb_test,
    "Wavelet+PCA": X_wav_test, "Combined+PCA": X_comb_test,
}

def run_experiment(feat_tr, feat_te, y_tr, y_te, groups, models, n_folds, task):
    """GroupKFold CV + final test evaluation for every feature-model combination."""
    gkf = GroupKFold(n_splits=n_folds)
    results = []
    preds = {}

    for feat_name in feat_tr:
        Xtr, Xte = feat_tr[feat_name], feat_te[feat_name]
        apply_pca = "+PCA" in feat_name
        for model_name, template in models.items():
            t0 = time.time()
            cv_f1, cv_f1_tr = [], []

            for ti, vi in gkf.split(Xtr, y_tr, groups=groups):
                sc = StandardScaler()
                Xti = sc.fit_transform(Xtr[ti])
                Xvi = sc.transform(Xtr[vi])

                if apply_pca:
                    pca_fold = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
                    Xti = pca_fold.fit_transform(Xti)
                    Xvi = pca_fold.transform(Xvi)

                clf = clone(template)

                if model_name == "XGBoost" and len(np.unique(y_tr[ti])) == 2:
                    n_neg = np.sum(y_tr[ti] == 0)
                    n_pos = np.sum(y_tr[ti] == 1)
                    clf.set_params(scale_pos_weight=n_neg / max(n_pos, 1))

                clf.fit(Xti, y_tr[ti])
                cv_f1.append(f1_score(y_tr[vi], clf.predict(Xvi), average="macro"))
                cv_f1_tr.append(f1_score(y_tr[ti], clf.predict(Xti), average="macro"))

            sc = StandardScaler()
            Xtr_s = sc.fit_transform(Xtr)
            Xte_s = sc.transform(Xte)

            if apply_pca:
                pca_final = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
                Xtr_s = pca_final.fit_transform(Xtr_s)
                Xte_s = pca_final.transform(Xte_s)

            clf_f = clone(template)
            if model_name == "XGBoost" and len(np.unique(y_tr)) == 2:
                n_neg = np.sum(y_tr == 0)
                n_pos = np.sum(y_tr == 1)
                clf_f.set_params(scale_pos_weight=n_neg / max(n_pos, 1))
            clf_f.fit(Xtr_s, y_tr)
            y_pred = clf_f.predict(Xte_s)
            test_f1  = f1_score(y_te, y_pred, average="macro")
            test_acc = accuracy_score(y_te, y_pred)

            preds[f"{feat_name}|{model_name}"] = y_pred
            results.append({
                "Task": task, "Features": feat_name, "Model": model_name,
                "CV F1 Mean": round(np.mean(cv_f1), 4),
                "CV F1 Std":  round(np.std(cv_f1), 4),
                "Train F1":   round(np.mean(cv_f1_tr), 4),
                "Test F1":    round(test_f1, 4),
                "Test Acc":   round(test_acc, 4),
            })
            elapsed = time.time() - t0
            pca_info = f"  (PCA: {Xtr_s.shape[1]}D)" if apply_pca else ""
            print(f"  [{task:5s}] {feat_name:16s} + {model_name:20s} -> "
                  f"CV F1={np.mean(cv_f1):.4f}+/-{np.std(cv_f1):.4f}  "
                  f"Test F1={test_f1:.4f}  ({elapsed:.0f}s){pca_info}")

    return pd.DataFrame(results), preds

In [ ]:
y_binary_train = (y_train != 0).astype(int)
y_binary_test  = (y_test  != 0).astype(int)

print(f"Binary label balance — Train: {np.bincount(y_binary_train)} | "
      f"Test: {np.bincount(y_binary_test)}")
print()
print("Running binary classification experiments ...")
print()
binary_results, binary_preds = run_experiment(
    feat_train, feat_test, y_binary_train, y_binary_test,
    group_ids_train, sup_models_binary, N_FOLDS, "Binary"
)

In [ ]:
print("=== Binary Classification Results ===")
print()
display(binary_results.sort_values("Test F1", ascending=False)
        .drop(columns=["Task"])
        .reset_index(drop=True)
        .style.format({"CV F1 Mean": "{:.4f}", "CV F1 Std": "{:.4f}",
                        "Train F1": "{:.4f}", "Test F1": "{:.4f}",
                        "Test Acc": "{:.4f}"}))

## 7) Supervised Modeling — Multi-class Classification (21 classes)

Target: `faultNumber` ∈ {0, 1, ..., 20}. Same evaluation protocol as binary.

In [ ]:
print("Running multi-class classification experiments ...")
print()
mc_results, mc_preds = run_experiment(
    feat_train, feat_test, y_train, y_test,
    group_ids_train, sup_models_multi, N_FOLDS, "Multi"
)

In [ ]:
print("=== Multi-class Classification Results ===")
print()
display(mc_results.sort_values("Test F1", ascending=False)
        .drop(columns=["Task"])
        .reset_index(drop=True)
        .style.format({"CV F1 Mean": "{:.4f}", "CV F1 Std": "{:.4f}",
                        "Train F1": "{:.4f}", "Test F1": "{:.4f}",
                        "Test Acc": "{:.4f}"}))

In [ ]:
best_row = mc_results.sort_values("Test F1", ascending=False).iloc[0]
best_key = f"{best_row['Features']}|{best_row['Model']}"
y_pred_best = mc_preds[best_key]

print(f"Best model: {best_row['Model']} with {best_row['Features']} features")
print(f"Test Macro F1 = {best_row['Test F1']:.4f}")
print()
print(classification_report(y_test, y_pred_best, digits=4, zero_division=0))

fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, ax=ax, cmap="Blues",
                                         normalize="true", values_format=".2f")
ax.set_title(f"Confusion Matrix — {best_row['Model']} ({best_row['Features']})")
plt.tight_layout()
plt.show()

In [ ]:
report = classification_report(y_test, y_pred_best, output_dict=True, zero_division=0)
per_fault = {int(k): v["f1-score"] for k, v in report.items() if k.isdigit()}
faults = sorted(per_fault.keys())
f1_vals = [per_fault[f] for f in faults]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#e74c3c" if f in [3, 9, 15] else "#3498db" for f in faults]
ax.bar([str(f) for f in faults], f1_vals, color=colors)
ax.set_xlabel("Fault Type")
ax.set_ylabel("F1-Score")
ax.set_title("Per-Fault F1 Score (Best Multi-class Model) — Red = Hard Faults (3, 9, 15)")
ax.axhline(y=np.mean(f1_vals), color="black", linestyle="--",
           label=f"Mean = {np.mean(f1_vals):.4f}")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### Run-Level Evaluation

Window-level metrics can overstate performance because many correlated windows from the same run contribute independently. Here we aggregate predictions to the run level using majority vote.

In [ ]:
def run_level_eval(y_true, y_pred, group_ids):
    """Aggregate window predictions to run level via majority vote."""
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "group": group_ids})
    agg = df.groupby("group").agg(
        true_label=("y_true", lambda x: x.mode()[0]),
        pred_label=("y_pred", lambda x: x.mode()[0]),
        n_windows=("y_true", "count"),
    ).reset_index()
    return agg

agg_test = run_level_eval(y_test, y_pred_best, group_ids_test)
run_f1 = f1_score(agg_test["true_label"], agg_test["pred_label"], average="macro")
run_acc = accuracy_score(agg_test["true_label"], agg_test["pred_label"])
print(f"Run-level evaluation (majority vote over windows per run):")
print(f"  Runs evaluated: {len(agg_test)}")
print(f"  Run-level Macro F1: {run_f1:.4f}")
print(f"  Run-level Accuracy: {run_acc:.4f}")
print(f"  (Window-level Macro F1 was {best_row['Test F1']:.4f})")

## 8) One-Class / Anomaly Detection

One-class models are trained **only on fault-free data** and evaluated on the mixed test set. The scaler is fit on fault-free training windows only. Thresholds are set using the 95th percentile of anomaly scores on fault-free training data — no test data influences the threshold.

**PCA addition:** PCA (95% variance) is also applied to Wavelet+PCA and Combined+PCA features in the one-class pipeline, fit on fault-free training data only.

### 8.1) Isolation Forest

In [ ]:
del binary_preds, mc_preds
gc.collect()

ff_mask_train = run_faults_train == 0
y_binary_test_oc = (y_test != 0).astype(int)

oc_results = []

for feat_name, Xtr_all, Xte_all in [
    ("Statistical", X_stats_train, X_stats_test),
    ("Wavelet",     X_wav_train,   X_wav_test),
    ("Combined",    X_comb_train,  X_comb_test),
    ("Wavelet+PCA", X_wav_train,   X_wav_test),
    ("Combined+PCA", X_comb_train, X_comb_test),
]:
    sc = StandardScaler()
    X_ff = sc.fit_transform(Xtr_all[ff_mask_train])
    X_te = sc.transform(Xte_all)

    if "+PCA" in feat_name:
        pca_oc = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
        X_ff = pca_oc.fit_transform(X_ff)
        X_te = pca_oc.transform(X_te)

    iso = IsolationForest(random_state=RANDOM_STATE, contamination="auto", n_jobs=-1)
    iso.fit(X_ff)

    scores_te = -iso.decision_function(X_te)
    scores_ff = -iso.decision_function(X_ff)
    threshold = np.percentile(scores_ff, 95)

    y_pred_iso = (scores_te > threshold).astype(int)
    iso_f1  = f1_score(y_binary_test_oc, y_pred_iso, average="macro")
    iso_acc = accuracy_score(y_binary_test_oc, y_pred_iso)
    try:
        iso_auc = roc_auc_score(y_binary_test_oc, scores_te)
    except ValueError:
        iso_auc = float("nan")

    oc_results.append({
        "Model": "Isolation Forest", "Features": feat_name,
        "F1": round(iso_f1, 4), "Accuracy": round(iso_acc, 4),
        "ROC-AUC": round(iso_auc, 4),
        "scores": scores_te, "y_true": y_binary_test_oc,
    })
    del sc, iso, X_ff, X_te, scores_ff, y_pred_iso
    gc.collect()
    print(f"  Isolation Forest + {feat_name:12s} -> F1={iso_f1:.4f}  AUC={iso_auc:.4f}")

### 8.2) Autoencoder

A symmetric autoencoder trained to reconstruct normal sensor feature vectors. High reconstruction error on test data indicates an anomaly. Architecture scales with input dimensionality.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        h1 = min(128, input_dim // 2)
        h2 = min(64, h1 // 2)
        h3 = min(32, h2 // 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2),        nn.ReLU(),
            nn.Linear(h2, h3),        nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(h3, h2),        nn.ReLU(),
            nn.Linear(h2, h1),        nn.ReLU(),
            nn.Linear(h1, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
AE_EPOCHS = 20
AE_LR     = 1e-3

torch.set_num_threads(1)
gc.collect()

def _ae_mse_batched(model, X_np, batch_size=2048):
    """Compute per-sample MSE in small batches — no large tensor copies."""
    errs = []
    for i in range(0, len(X_np), batch_size):
        chunk = torch.from_numpy(X_np[i:i+batch_size])
        recon = model(chunk)
        errs.append(torch.mean((chunk - recon) ** 2, dim=1).numpy())
        del chunk, recon
    return np.concatenate(errs)

for feat_name, Xtr_all, Xte_all in [
    ("Statistical", X_stats_train, X_stats_test),
    ("Wavelet",     X_wav_train,   X_wav_test),
    ("Combined",    X_comb_train,  X_comb_test),
    ("Wavelet+PCA", X_wav_train,   X_wav_test),
    ("Combined+PCA", X_comb_train, X_comb_test),
]:
    sc = StandardScaler()
    X_ff = sc.fit_transform(Xtr_all[ff_mask_train])

    if "+PCA" in feat_name:
        pca_oc = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
        X_ff = pca_oc.fit_transform(X_ff)

    X_ff = X_ff.astype(np.float32)
    input_dim = X_ff.shape[1]
    torch.manual_seed(RANDOM_STATE)
    model_ae = Autoencoder(input_dim)
    optimizer = torch.optim.Adam(model_ae.parameters(), lr=AE_LR)

    X_ff_t = torch.from_numpy(X_ff)
    print(f"  Training Autoencoder on {feat_name} ({input_dim}D, {len(X_ff)} samples) ...")
    for epoch in range(AE_EPOCHS):
        model_ae.train()
        out = model_ae(X_ff_t)
        loss = nn.functional.mse_loss(out, X_ff_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{AE_EPOCHS}  Loss={loss.item():.6f}")
    del X_ff_t, out, loss

    model_ae.eval()
    with torch.no_grad():
        err_ff = _ae_mse_batched(model_ae, X_ff)
    threshold = np.percentile(err_ff, 95)
    del X_ff, err_ff

    X_te = sc.transform(Xte_all)
    if "+PCA" in feat_name:
        X_te = pca_oc.transform(X_te)
    X_te = X_te.astype(np.float32)
    del sc
    with torch.no_grad():
        err_te = _ae_mse_batched(model_ae, X_te)
    del X_te, model_ae, optimizer
    gc.collect()

    y_pred_ae = (err_te > threshold).astype(int)
    ae_f1  = f1_score(y_binary_test_oc, y_pred_ae, average="macro")
    ae_acc = accuracy_score(y_binary_test_oc, y_pred_ae)
    try:
        ae_auc = roc_auc_score(y_binary_test_oc, err_te)
    except ValueError:
        ae_auc = float("nan")

    oc_results.append({
        "Model": "Autoencoder", "Features": feat_name,
        "F1": round(ae_f1, 4), "Accuracy": round(ae_acc, 4),
        "ROC-AUC": round(ae_auc, 4),
        "scores": err_te, "y_true": y_binary_test_oc,
    })
    del y_pred_ae
    gc.collect()
    print(f"  Autoencoder + {feat_name:12s} -> F1={ae_f1:.4f}  AUC={ae_auc:.4f}")
    print()

In [ ]:
oc_table = pd.DataFrame([{k: v for k, v in r.items() if k not in ("scores", "y_true")}
                          for r in oc_results])
print("=== One-Class / Anomaly Detection Results ===")
print()
display(oc_table.sort_values("F1", ascending=False).reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
styles = {"Isolation Forest": "--", "Autoencoder": "-"}

for r in oc_results:
    if r["Features"] in ("Combined", "Combined+PCA"):
        fpr, tpr, _ = roc_curve(r["y_true"], r["scores"])
        ax.plot(fpr, tpr, linestyle=styles[r["Model"]],
                label=f"{r['Features']} {r['Model']} (AUC={r['ROC-AUC']:.3f})")

ax.plot([0, 1], [0, 1], "k:", alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — One-Class Models (Combined Features ± PCA)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_oc = max(oc_results, key=lambda r: r["F1"])
print(f"Best one-class model: {best_oc['Model']} + {best_oc['Features']}")
print()

thresh_oc = np.percentile(best_oc["scores"][y_binary_test_oc == 0],
                           95) if len(best_oc["scores"][y_binary_test_oc == 0]) > 0 else np.median(best_oc["scores"])

unique_faults = sorted(np.unique(y_test))
det_rates = []
for fn in unique_faults:
    mask = y_test == fn
    if mask.sum() == 0:
        continue
    if fn == 0:
        fpr_val = np.mean(best_oc["scores"][mask] > thresh_oc)
        det_rates.append({"Fault": fn, "Windows": mask.sum(),
                         "Rate": round(fpr_val, 4), "Type": "FPR"})
    else:
        det = np.mean(best_oc["scores"][mask] > thresh_oc)
        det_rates.append({"Fault": fn, "Windows": mask.sum(),
                         "Rate": round(det, 4), "Type": "Detection Rate"})

det_df = pd.DataFrame(det_rates)
print("Per-fault detection rates:")
print(det_df.to_string(index=False))

## 9) Evaluation and Comparison

Unified comparison across all models, feature sets, and tasks.

In [ ]:
all_results = pd.concat([binary_results, mc_results], ignore_index=True)

print("=== Unified Supervised Results ===")
print()
display(all_results.sort_values(["Task", "Test F1"], ascending=[True, False])
        .reset_index(drop=True)
        .style.format({"CV F1 Mean": "{:.4f}", "CV F1 Std": "{:.4f}",
                        "Train F1": "{:.4f}", "Test F1": "{:.4f}",
                        "Test Acc": "{:.4f}"}))

In [ ]:
for task in ["Binary", "Multi"]:
    sub = all_results[all_results["Task"] == task]
    fig, ax = plt.subplots(figsize=(14, 6))
    models = sub["Model"].unique()
    feats  = sub["Features"].unique()
    width = 0.8 / len(feats)
    x = np.arange(len(models))

    for i, feat in enumerate(feats):
        vals = [sub[(sub["Model"] == m) & (sub["Features"] == feat)]["Test F1"].values[0]
                for m in models]
        ax.bar(x + i * width, vals, width, label=feat, alpha=0.8)

    ax.set_xlabel("Model")
    ax.set_ylabel("Test Macro F1")
    ax.set_title(f"{task} Classification — Test F1 by Model and Feature Set")
    ax.set_xticks(x + width * (len(feats) - 1) / 2)
    ax.set_xticklabels(models, rotation=15)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for idx, task in enumerate(["Binary", "Multi"]):
    sub = all_results[all_results["Task"] == task]
    labels_plot = [f"{r['Features'][:4]}+{r['Model'][:3]}" for _, r in sub.iterrows()]
    axes[idx].scatter(sub["Train F1"], sub["Test F1"], s=80)
    for i, lbl in enumerate(labels_plot):
        axes[idx].annotate(lbl, (sub.iloc[i]["Train F1"], sub.iloc[i]["Test F1"]),
                          fontsize=6, alpha=0.7)
    axes[idx].plot([0.4, 1], [0.4, 1], "k--", alpha=0.3)
    axes[idx].set_xlabel("Train F1 (CV mean)")
    axes[idx].set_ylabel("Test F1")
    axes[idx].set_title(f"{task} — Train vs Test F1 (Overfitting Diagnostic)")
    axes[idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Average Test F1 by Feature Set ===")
print()
for task in ["Binary", "Multi"]:
    sub = all_results[all_results["Task"] == task]
    avg = sub.groupby("Features")["Test F1"].mean().sort_values(ascending=False)
    print(f"{task}:")
    for feat, val in avg.items():
        print(f"  {feat:12s} {val:.4f}")
    print()

print("=== Best Configuration per Task ===")
print()
for task in ["Binary", "Multi"]:
    sub = all_results[all_results["Task"] == task]
    best = sub.sort_values("Test F1", ascending=False).iloc[0]
    print(f"  {task:5s}: {best['Model']:20s} + {best['Features']:12s} "
          f"-> Test F1 = {best['Test F1']:.4f}")

print()
if len(oc_results) > 0:
    best_oc_row = max(oc_results, key=lambda r: r["F1"])
    print(f"  Best one-class: {best_oc_row['Model']:20s} + {best_oc_row['Features']:12s} "
          f"-> Test F1 = {best_oc_row['F1']:.4f}")

## 10) Conclusions

### Summary of Findings

This notebook adds PCA (95% explained variance) to the Wavelet and Combined feature pipelines. The Statistical baseline is unchanged. Key observations:

1. **Feature pipeline comparison:** The relative performance of statistical, wavelet, and combined feature sets varies by model and task. The combined set does not universally dominate — for some models, simpler feature sets perform comparably or better, likely due to the curse of dimensionality.

2. **PCA dimensionality reduction:** PCA was applied to Wavelet and Combined features after scaling, fit only on training data within each CV fold. The retained dimensionality and its effect on F1 can be compared against the non-PCA baselines above.

3. **Supervised models:** Tree-based models (Random Forest, XGBoost) generally achieve the highest macro F1 scores on this tabular feature data, with XGBoost typically leading. Logistic Regression provides a useful linear baseline.

4. **Hard faults (3, 9, 15):** These fault types, well-documented in the TEP literature as difficult to detect, show lower per-fault F1 scores. This is consistent with prior work and reflects the subtlety of these fault signatures.

5. **One-class detection:** Isolation Forest and Autoencoder models trained only on fault-free data provide anomaly detection capability, though they underperform supervised models as expected since they cannot leverage fault-type information. The per-fault detection rate table reveals which fault types are most detectable from normal behavior deviation alone.

6. **Run-level vs window-level metrics:** Run-level evaluation (via majority vote) provides a more realistic assessment of operational performance than window-level metrics, which inflate sample sizes through correlated overlapping windows.

7. **Signal-quality features:** The inclusion of coefficient of variation, lag-1 autocorrelation, first-difference std, and high-frequency energy ratio alongside traditional statistical features provides complementary information about sensor signal behavior within each window.

### Limitations

- **TEP faults are process disturbances, not sensor degradation.** The results reflect process fault classification performance. True sensor-specific degradation detection would require a different dataset or synthetic fault injection.
- **Subsampled dataset:** Using 30 runs per fault type for tractable runtime means results have wider confidence intervals than the full 500-run dataset.
- **No hyperparameter tuning:** Default hyperparameters are used throughout. Tuned models may perform better.
- **Window-level evaluation remains primary.** Run-level analysis is supplementary. Detection delay and event-level analysis are not evaluated.
- **Measured variables only.** Manipulated variables (xmv) are excluded to avoid confounding from controller responses. Including them might improve raw classification accuracy but at the cost of methodological clarity.

### Future Work

- Explore deeper neural architectures (1D-CNN, LSTM) operating directly on raw time series.
- Test additional wavelet families and decomposition levels.
- Experiment with different PCA variance thresholds or kernel PCA for non-linear dimensionality reduction.
- Investigate ensemble methods combining supervised and one-class detectors.
- Apply to real industrial sensor data to validate transferability.
- Incorporate detection delay analysis for practical deployment evaluation.